In [ ]:
# Cargar librerías
library(tidyverse)
library(lubridate)

In [ ]:
# Leer datos
datos = read_csv("llegadas_restaurante.csv")

In [ ]:
# Gráfico de barras para el tamaño de los grupos
ggplot(data=datos) +
  geom_bar(mapping=aes(x=factor(tamano)), color="black", fill="yellow") +
  labs(title = "Distribución del tamaño de grupos",
       x = "Tamaño del grupo",
       y = "Frecuencia") +
  theme_minimal()

In [ ]:
# Convertir la columna hora (formato HH:MM)
datos <- datos %>%
  mutate(hora = as.POSIXct(hora, format = "%H:%M", tz = "UTC")) %>%
  arrange(hora) %>%
  mutate(tiempo_entre_llegadas = as.numeric(difftime(hora, lag(hora), units = "mins"))) %>%
  drop_na(tiempo_entre_llegadas)

In [ ]:
# Resumen estadístico
summary(datos$tiempo_entre_llegadas)

cat("Media:", mean(datos$tiempo_entre_llegadas), "\n")
cat("Número de observaciones:", nrow(datos), "\n")
cat("Desviación estándar:", sd(datos$tiempo_entre_llegadas), "\n")

In [ ]:
# parámetros empíricos

beta = mean(datos$tiempo_entre_llegadas)
n_obs = nrow(datos)

In [ ]:
# ========================
# Gráfico QQ para distribución Exponencial
# ========================

# Calcular cuantiles teóricos de la exponencial
cuantiles_teoricos <- qexp(ppoints(n_obs),
                           rate = 1 / beta)

# Gráfico QQ
ggplot() +
  geom_point(aes(x = cuantiles_teoricos, y = sort(datos$tiempo_entre_llegadas))) +
  geom_abline(intercept = 0, slope = 1, color = "red", linetype = "dashed") +
  labs(title = "Gráfico QQ - Tiempos entre llegadas vs Exponencial",
       x = "Cuantiles teóricos (Exponencial)",
       y = "Cuantiles observados") +
  theme_minimal()


In [ ]:
# ========================
# Análisis adicional: Histograma con curva exponencial
# ========================

# Histograma de los tiempos entre llegadas
ggplot(datos, aes(x = tiempo_entre_llegadas)) +
  geom_histogram(aes(y = after_stat(density)), bins = 30,
                 color = "black", fill = "yellow", alpha = 0.7) +
  stat_function(fun = dexp,
                args = list(rate = 1/beta),
                color = "red", linewidth = 1.2) +
  labs(title = "Histograma de tiempos entre llegadas",
       subtitle = "Con curva teórica exponencial superpuesta",
       x = "Tiempo entre llegadas (minutos)",
       y = "Densidad") +
  theme_minimal()